# Week 10: Reproducible ML Workflow Studio
Machine Learning in Engineering Science / NCKU + LAiMM / Fall 2026

**Monday, 9 November 2026, 14:00-17:00.** H1: 45 min explanation + 15 min practice; H2: 15 + 45; H3: 60 min lab. Total: **60 min explanation, 120 min practice**. Eight core exercises and two optional challenges; no new graded weekly homework.

**What you will demonstrate:** turn an analysis into a portable evidence package that another process can load, validate, refit and compare by stable sample ID. Week 9 diagnosed reliability; this week tests the workflow used to support that diagnosis.

**Tools:** free Colab CPU or Python 3 Jupyter. No paid AI, GPU, API key or remote dataset download. scikit-learn's bundled diabetes regression dataset is retained from the original Week 10 material, using **raw inputs (`scaled=False`)** so all fitted preprocessing is training-only. This is a statistical teaching exercise, not clinical guidance, diagnosis, causal discovery or validation of a medical tool.

Read each prediction prompt before running. Run cells in order. If using the exercise notebook, its opening walkthrough is already complete: do not also run a second copy of the practice notebook.

## 1. Set up the environment and declare choices
We use the packaged 442-row, 10-feature regression dataset. We do not interpret individual features medically. Source IDs such as `diabetes-0000` identify rows in this frozen dataset version, not globally stable patient identities.

Declare four candidates now: a training-target median baseline, and training-scaled Ridge with alpha 1, 20, or 100. Select by validation MAE; exact ties follow the listed candidate order. Do not choose a new winner after opening final-test results.

**Predict:** is a random seed sufficient if someone rearranges the source rows before calling the same split function?

In [ ]:
import os,sys,subprocess,importlib.util,platform,json,csv,hashlib,zipfile,tempfile,shutil
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'1')
required={'numpy':'numpy','matplotlib':'matplotlib','sklearn':'scikit-learn'}
missing=[p for m,p in required.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib,matplotlib.pyplot as plt
import sklearn,scipy
from sklearn.datasets import load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
CONFIG={'seed':2026,'dataset':'sklearn diabetes raw; statistical teaching only',
 'train_n':265,'validation_n':88,'test_n':89,'alphas':[1.,20.,100.],
 'candidate_order':['median','ridge-1','ridge-20','ridge-100'],
 'selection':'validation MAE; exact tie follows candidate_order','atol':1e-8,'rtol':1e-10}
ENV={'python':platform.python_version(),'numpy':np.__version__,'scikit_learn':sklearn.__version__,'matplotlib':matplotlib.__version__,'scipy':scipy.__version__,'platform':platform.platform()}
RUN_LABEL='reference'
ARTIFACTS=Path('artifacts')/('week10-'+RUN_LABEL);ARTIFACTS.mkdir(parents=True,exist_ok=True)
BUNDLE_DIR=ARTIFACTS/'handover';BUNDLE_DIR.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize':(10,6.9),'savefig.dpi':160,'font.size':22,
 'axes.labelsize':22,'axes.titlesize':24,'xtick.labelsize':17,'ytick.labelsize':18,
 'legend.fontsize':17,'axes.spines.top':False,'axes.spines.right':False})
TEAL='#167b87';RED='#7f2635';GOLD='#b88a29'
def finish(fig,name):
 fig.savefig(ARTIFACTS/name,bbox_inches='tight',facecolor='white');plt.show();plt.close(fig)
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def metrics(y,p):return {'n':int(len(y)),'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
def candidate(name):
 if name=='median':return DummyRegressor(strategy='median')
 return make_pipeline(StandardScaler(),Ridge(alpha=float(name.split('-')[1]),solver='svd'))
print('Actual environment:',ENV);print('Declared configuration:',CONFIG)

## 2. Freeze row identity, measurements and partition membership
Load raw features, not the dataset loader's globally pre-scaled defaults. Generate source IDs once. Draw the seeded permutation of these IDs and reserve 265 train, 88 validation and 89 test rows. The three sets are disjoint and exhaustive.

We use a random-row split only as a fixed statistical teaching protocol. This does not establish external, chronological, grouped or clinical generalization. A row ID is not a feature.

**Action:** inspect column names and all counts before fitting. If any check fails, stop; do not delete the assertion to obtain a score.

In [ ]:
data=load_diabetes(scaled=False)
X=np.asarray(data.data,dtype=float);y=np.asarray(data.target,dtype=float)
FEATURES=list(data.feature_names);IDS=np.array([f'diabetes-{i:04d}' for i in range(len(y))])
assert X.shape==(442,10) and y.shape==(442,)
assert np.isfinite(X).all() and np.isfinite(y).all()
order=np.random.default_rng(CONFIG['seed']).permutation(len(y))
train_idx=np.sort(order[:265]);val_idx=np.sort(order[265:353]);test_idx=np.sort(order[353:])
SPLITS={k:IDS[idx].tolist() for k,idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]}
assert all(not(set(SPLITS[a])&set(SPLITS[b])) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert set(sum(SPLITS.values(),[]))==set(IDS)
Xtr,ytr=X[train_idx],y[train_idx];Xva,yva=X[val_idx],y[val_idx];Xte,yte=X[test_idx],y[test_idx]
HEADER=['row_id',*FEATURES,'target']
DATA_PATH=BUNDLE_DIR/'frozen.csv'
with DATA_PATH.open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(HEADER)
 for i in range(len(y)):w.writerow([IDS[i],*[repr(float(v)) for v in X[i]],repr(float(y[i]))])
(BUNDLE_DIR/'split.json').write_text(json.dumps(SPLITS,indent=2),encoding='utf-8')
data_sha256=digest(DATA_PATH)
print('Rows / features:',X.shape,'Columns:',HEADER)
print('Partition sizes:',{k:len(v) for k,v in SPLITS.items()});print('CSV SHA-256:',data_sha256)
fig,ax=plt.subplots(layout='constrained')
for name,idx,c in [('train',train_idx,TEAL),('validation',val_idx,GOLD),('test',test_idx,RED)]:
 ax.hist(y[idx],bins=np.linspace(y.min(),y.max(),15),density=True,histtype='step',lw=2,label=name+' n='+str(len(idx)),color=c)
ax.set(xlabel='Packaged regression target',ylabel='Empirical density');ax.legend();finish(fig,'w10-target-distributions.png')

## 3. Make the data contract executable
The contract checks exact column order, unique IDs, finite numeric values, the expected row count, split sizes and a disjoint exhaustive partition. It rejects a wrong interface instead of silently repairing it.

**Predict:** if you shuffle CSV rows but keep IDs and split lists, should selected training membership change? If you shuffle feature columns without a declared adapter, should the strict interface accept it?

The readable runner below is the program another process will execute. It fits only the already selected candidate. It does not inspect validation scores to reselect and does not load a pickle. Only run source code that you trust and have reviewed; a checksum manifest does not authenticate an unknown author.

In [ ]:
RUNNER = "import argparse,csv,json,hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.dummy import DummyRegressor\nfrom sklearn.metrics import mean_absolute_error,mean_squared_error\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef read_contract(root,config):\n    with (root/'frozen.csv').open(newline='',encoding='utf-8') as f:\n        reader=csv.DictReader(f)\n        if reader.fieldnames!=['row_id',*config['features'],'target']:raise ValueError('schema: exact column order required')\n        rows=list(reader)\n    if len(rows)!=config['row_count']:raise ValueError('schema: row count')\n    if any(None in r or any(v is None for v in r.values()) for r in rows):raise ValueError('schema: row width')\n    ids=[r['row_id'] for r in rows]\n    if any(not i.strip() for i in ids):raise ValueError('schema: empty row ID')\n    if len(set(ids))!=len(ids):raise ValueError('schema: duplicate row ID')\n    values=np.array([[float(r[k]) for k in [*config['features'],'target']] for r in rows])\n    if not np.isfinite(values).all():raise ValueError('schema: non-finite numeric value')\n    splits=json.loads((root/'split.json').read_text())\n    if set(splits)!={'train','validation','test'}:raise ValueError('schema: split names')\n    flat=sum(splits.values(),[])\n    if len(flat)!=len(set(flat)) or set(flat)!=set(ids):raise ValueError('schema: split overlap or missing IDs')\n    if {k:len(v) for k,v in splits.items()}!=config['split_counts']:raise ValueError('schema: split counts')\n    lookup={r['row_id']:r for r in rows}\n    def cohort(name):\n        subset=[lookup[i] for i in splits[name]]\n        return np.array([[float(r[k]) for k in config['features']] for r in subset]),np.array([float(r['target']) for r in subset])\n    return cohort,splits\n\ndef run(root,out):\n    root=Path(root).resolve()\n    manifest=json.loads((root/'manifest.json').read_text())\n    expected={'frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json'}\n    if set(manifest['sha256'])!=expected:raise ValueError('manifest: required file list')\n    for name,value in manifest['sha256'].items():\n        if digest(root/name)!=value:raise ValueError('checksum mismatch: '+name)\n    config=json.loads((root/'config.json').read_text())\n    cohort,splits=read_contract(root,config)\n    Xtr,ytr=cohort('train');Xte,yte=cohort('test')\n    chosen=config['selected']\n    if chosen not in config['candidate_order']:raise ValueError('config: undeclared selected candidate')\n    model=DummyRegressor(strategy='median') if chosen=='median' else make_pipeline(StandardScaler(),Ridge(alpha=float(chosen.split('-')[1]),solver='svd'))\n    model.fit(Xtr,ytr);pred=model.predict(Xte)\n    result={'selected':chosen,'data_sha256':digest(root/'frozen.csv'),'test_ids':splits['test'],'predictions':pred.tolist(),\n       'metrics':{'n':len(yte),'MAE':float(mean_absolute_error(yte,pred)),'RMSE':float(np.sqrt(mean_squared_error(yte,pred)))}}\n    Path(out).write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')\n    print(json.dumps({'selected':chosen,'test_n':len(yte),'output_written':True}))\nif __name__=='__main__':\n    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--out',required=True);a=p.parse_args();run(a.root,a.out)\n"
(BUNDLE_DIR/'replay.py').write_text(RUNNER,encoding='utf-8')
print('Readable runner exported:',BUNDLE_DIR/'replay.py')

## 4. Fit only on training data; choose on validation
The median of training targets is an interpretable constant baseline for absolute error. The Ridge candidates minimize a squared-error objective plus an L2 coefficient penalty, with an unpenalized intercept in scikit-learn. Selection still uses validation MAE; the training objective and evaluation metric need not be identical.

**Predict:** will stronger regularization always improve validation MAE? Why should target medians and feature means come only from training rows?

In [ ]:
models={name:candidate(name).fit(Xtr,ytr) for name in CONFIG['candidate_order']}
validation={name:metrics(yva,m.predict(Xva)) for name,m in models.items()}
selected=min(CONFIG['candidate_order'],key=lambda name:(validation[name]['MAE'],CONFIG['candidate_order'].index(name)))
print('Validation:',json.dumps(validation,indent=2));print('Selected before final test:',selected)
fig,ax=plt.subplots(layout='constrained')
ax.bar(range(4),[validation[n]['MAE'] for n in models],color=[TEAL if n==selected else RED for n in models])
ax.set_xticks(range(4),['Median','Ridge 1','Ridge 20','Ridge 100']);ax.set_ylabel('Validation MAE (target units)');finish(fig,'w10-validation-selection.png')
fig,ax=plt.subplots(layout='constrained');alphas=CONFIG['alphas']
norms=[float(np.linalg.norm(models['ridge-'+f'{a:g}'].named_steps['ridge'].coef_)) for a in alphas]
ax.plot(alphas,norms,'o-',color=TEAL,lw=2);ax.set_xscale('log');ax.set(xlabel='Ridge alpha',ylabel='Standardized coefficient L2 norm');finish(fig,'w10-regularization-path.png')

## 5. Open the final test once
Report the validation-selected candidate and the predeclared median baseline on the same 89 test observations. For diagnosis, inspect parity, residuals (`prediction - observation`) and the absolute-error CDF. A successful replay later does not make an inaccurate or externally invalid model useful.

**Action:** identify one actual large-residual sample ID. Do not tune alpha after seeing this test result.

In [ ]:
test_prediction={n:models[n].predict(Xte) for n in dict.fromkeys(['median',selected])}
test_metrics={n:metrics(yte,p) for n,p in test_prediction.items()}
pred=test_prediction[selected];residual=pred-yte
print('Final test:',json.dumps(test_metrics,indent=2))
fig,ax=plt.subplots(layout='constrained');names=list(test_metrics);width=.34;xx=np.arange(len(names))
for j,m in enumerate(['MAE','RMSE']):ax.bar(xx+(j-.5)*width,[test_metrics[n][m] for n in names],width,label=m)
ax.set_xticks(xx,names);ax.set_ylabel('Target units');ax.legend();finish(fig,'w10-test-metrics.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(yte,pred,s=36,alpha=.8,color=TEAL)
limits=[float(min(yte.min(),pred.min())),float(max(yte.max(),pred.max()))];ax.plot(limits,limits,'k--',lw=1)
ax.set(xlabel='Observed test target',ylabel='Selected-model prediction');finish(fig,'w10-test-parity.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(pred,residual,s=36,color=RED,alpha=.8);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Selected-model prediction',ylabel='Prediction - observation');finish(fig,'w10-test-residuals.png')
fig,ax=plt.subplots(layout='constrained')
for n,p in test_prediction.items():
 e=np.sort(np.abs(p-yte));ax.plot(e,np.arange(1,len(e)+1)/len(e),label=n,lw=2)
ax.set(xlabel='Absolute error (target units)',ylabel='Empirical cumulative fraction');ax.legend();finish(fig,'w10-error-cdf.png')
worst=np.argsort(np.abs(residual))[-5:][::-1]
worst_rows=[{'row_id':IDS[test_idx[j]],'observed':float(yte[j]),'predicted':float(pred[j]),'residual':float(residual[j])} for j in worst]
print('Five largest absolute residuals:',json.dumps(worst_rows,indent=2))

## 6. Interpret coordinates, not clinical causes
For a Ridge pipeline, the coefficient on standardized feature j is the model's response change for +1 **training** standard deviation of that feature, all other inputs fixed. This is not causal importance; correlated features can trade coefficients.

Recover the raw equation with `raw_coef = coef / scale` and `raw_intercept = intercept - dot(raw_coef, mean)`. Verify predictions. If the selected candidate is the median, its response is constant and this feature interpretation is not applicable; we then inspect the predeclared Ridge-20 candidate separately.

**Predict:** should shuffling feature order without matching names preserve the model's meaning?

In [ ]:
explain_name=selected if selected!='median' else 'ridge-20'
explain_model=models[explain_name];scaler=explain_model.named_steps['standardscaler'];reg=explain_model.named_steps['ridge']
raw_coef=reg.coef_/scaler.scale_;raw_intercept=float(reg.intercept_-np.dot(raw_coef,scaler.mean_))
np.testing.assert_allclose(raw_intercept+Xte@raw_coef,explain_model.predict(Xte),atol=1e-8,rtol=1e-10)
coefficient_record={'candidate':explain_name,'features':FEATURES,'standardized':reg.coef_.tolist(),
 'raw':raw_coef.tolist(),'raw_intercept':raw_intercept,'training_mean':scaler.mean_.tolist(),'training_scale':scaler.scale_.tolist()}
fig,ax=plt.subplots(layout='constrained');ax.barh(FEATURES,reg.coef_,color=[TEAL if v>=0 else RED for v in reg.coef_]);ax.axvline(0,color='black',lw=1)
ax.set(xlabel='Response change per +1 training SD',ylabel='Input coordinate',title=explain_name+' / function, not causation');finish(fig,'w10-coefficients.png')
mean_gap=(X.mean(axis=0)-Xtr.mean(axis=0))/Xtr.std(axis=0)
fig,ax=plt.subplots(layout='constrained');ax.bar(FEATURES,mean_gap,color=RED);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Feature',ylabel='(All-row mean - train mean) / train SD');finish(fig,'w10-preprocessing-scope.png')

## 7. Publish a small, readable handover contract
The handover contains exact measurements, split IDs, feature order, selected configuration, training procedure, reference test predictions and the result record. Store actual runtime versions. The manifest detects file changes relative to its recorded digests; anyone able to replace both files and digests can defeat it. It is not a signature or a scientific validity test.

No model pickle is needed: we refit from readable source and frozen inputs. Do not execute unfamiliar code or load an untrusted pickle simply because it has a checksum.

In [ ]:
FROZEN_CONFIG={**CONFIG,'features':FEATURES,'row_count':442,'split_counts':{'train':265,'validation':88,'test':89},'selected':selected,'environment':ENV}
(BUNDLE_DIR/'config.json').write_text(json.dumps(FROZEN_CONFIG,indent=2),encoding='utf-8')
with (BUNDLE_DIR/'reference-predictions.csv').open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(['row_id','observed','predicted'])
 for i,actual,p in zip(IDS[test_idx],yte,pred):w.writerow([i,repr(float(actual)),repr(float(p))])
record={'week':10,'data_scope':'Packaged regression dataset; statistical teaching only, not clinical validation',
 'config':CONFIG,'environment':ENV,'data_sha256':data_sha256,'selected_on_validation':selected,
 'validation':validation,'test':test_metrics,'coefficients':coefficient_record,'worst_test_rows':worst_rows,
 'claim_limit':'Frozen random-row teaching split; no external clinical, causal or deployment claim.'}
(BUNDLE_DIR/'result.json').write_text(json.dumps(record,indent=2,allow_nan=False),encoding='utf-8')
BUNDLE_FILES=['frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json']
manifest={'format':'MLES-W10-v1','sha256':{name:digest(BUNDLE_DIR/name) for name in BUNDLE_FILES}}
(BUNDLE_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Handover files:',BUNDLE_FILES+['manifest.json'])

## 8. Replay in a genuinely separate process
The next cell starts a new Python interpreter in a new working directory. It receives only the handover path and an output path. Notebook variables and fitted model objects are not shared. This is stronger than refitting in a new variable, but it still uses the same installed software environment and dataset; it is not independent scientific replication.

**Action:** inspect `handover/replay.py` before running it. The file was generated above in this notebook; do not substitute downloaded untrusted code.

**Stopping check:** exact test-ID sequence plus prediction agreement under `atol=1e-8`, `rtol=1e-10`. Investigate discrepancies before relaxing tolerances.

In [ ]:
REPLAY_PATH=(ARTIFACTS/'fresh-process-result.json').resolve()
with tempfile.TemporaryDirectory(prefix='mles-w10-replay-') as work:
 completed=subprocess.run([sys.executable,str((BUNDLE_DIR/'replay.py').resolve()),'--root',str(BUNDLE_DIR.resolve()),'--out',str(REPLAY_PATH)],cwd=work,capture_output=True,text=True,timeout=120)
 if completed.returncode:raise RuntimeError(completed.stderr)
 print('Fresh process:',completed.stdout.strip())
replay=json.loads(REPLAY_PATH.read_text())
assert replay['test_ids']==SPLITS['test'] and replay['data_sha256']==data_sha256
replay_diff=np.asarray(replay['predictions'])-pred
np.testing.assert_allclose(replay['predictions'],pred,atol=CONFIG['atol'],rtol=CONFIG['rtol'])
max_replay_difference=float(np.max(np.abs(replay_diff)))
print('Maximum prediction difference:',max_replay_difference)
fig,ax=plt.subplots(layout='constrained');ax.scatter(np.arange(len(pred)),replay_diff,color=TEAL,s=30);ax.axhline(0,color='black',lw=1)
ax.set_ylim(-1e-7,1e-7);ax.set(xlabel='Test observation in saved ID order',ylabel='Fresh process - notebook prediction');finish(fig,'w10-replay-differences.png')
# Import the reviewed runner functions without running its CLI main block.
import importlib.util
spec=importlib.util.spec_from_file_location('week10_reviewed_runner',BUNDLE_DIR/'replay.py')
runner_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(runner_module)

## 9. Test the contract's rejection behavior
A good handover also says what it will refuse. We deliberately corrupt temporary copies, never the reference folder. Duplicated IDs, a non-finite numeric value and reordered columns should be rejected by the strict data contract. These are interface tests, not new performance evaluations.

**Predict:** if a failure is correctly caught, should the notebook itself crash? No: the cell catches the expected error and records the rejection. An unexpected acceptance is a failed check.

In [ ]:
def contract_probe(kind):
 with tempfile.TemporaryDirectory(prefix='mles-w10-probe-') as work:
  target=Path(work);shutil.copy2(DATA_PATH,target/'frozen.csv');shutil.copy2(BUNDLE_DIR/'split.json',target/'split.json')
  with (target/'frozen.csv').open(newline='',encoding='utf-8') as f:rows=list(csv.reader(f))
  if kind=='duplicate ID':rows[2][0]=rows[1][0]
  elif kind=='extra value':rows[1].append('unexpected')
  elif kind=='missing value':rows[1].pop()
  elif kind=='empty ID':rows[1][0]=''
  elif kind=='non-finite':rows[1][1]='nan'
  elif kind=='column order':rows=[[r[1],r[0],*r[2:]] for r in rows]
  else:raise ValueError(kind)
  with (target/'frozen.csv').open('w',newline='',encoding='utf-8') as f:csv.writer(f).writerows(rows)
  try:runner_module.read_contract(target,FROZEN_CONFIG)
  except ValueError as e:return {'case':kind,'rejected':True,'message':str(e)}
  return {'case':kind,'rejected':False,'message':'Unexpected acceptance'}
contract_checks=[contract_probe(k) for k in ['duplicate ID','non-finite','column order']]
assert all(r['rejected'] for r in contract_checks)
fig,ax=plt.subplots(layout='constrained');ax.barh([r['case'] for r in contract_checks],[int(r['rejected']) for r in contract_checks],color=TEAL)
ax.set_xlim(0,1.2);ax.set_xticks([0,1],['Accepted','Rejected']);ax.set_xlabel('Actual contract-test outcome');finish(fig,'w10-contract-tests.png')
print(json.dumps(contract_checks,indent=2));assert digest(DATA_PATH)==data_sha256

## 10. Save the bundle, limitation and explain-back
Download the ZIP from Colab's Files panel before ending the runtime. The handover can be extracted and replayed in another compatible Python environment using its readable runner. Review code before execution. Never treat a passing rerun as a guarantee of external validity.

Your two-minute explanation should identify: the prediction task, actual baseline comparison, one failure ID, exact data/configuration identity, what the fresh process established, and what it did not establish. Written and oral midterm examinations remain unchanged; this lab is preparation, not a new graded assignment.

In [ ]:
record.update({'replay_max_prediction_difference':max_replay_difference,'contract_tests':contract_checks,
 'replay_scope':'Fresh interpreter and working directory; same installed environment, not independent replication'})
(ARTIFACTS/'week10-evidence.json').write_text(json.dumps(record,indent=2,allow_nan=False),encoding='utf-8')
HANDOVER_ZIP=ARTIFACTS/'week10-workflow-handover.zip'
with zipfile.ZipFile(HANDOVER_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for n in BUNDLE_FILES+['manifest.json']:z.write(BUNDLE_DIR/n,arcname='handover/'+n)
 for p in sorted(ARTIFACTS.glob('w10-*.png')):z.write(p,arcname='figures/'+p.name)
 z.write(ARTIFACTS/'week10-evidence.json',arcname='week10-evidence.json')
 z.write(REPLAY_PATH,arcname='fresh-process-result.json')
with zipfile.ZipFile(HANDOVER_ZIP) as z:assert z.testzip() is None
print('Download:',HANDOVER_ZIP,'Bytes:',HANDOVER_ZIP.stat().st_size)
print('After extraction and code review: python handover/replay.py --root handover --out replay-result.json')

### References
- [Dataset API and raw/scaled input contract](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
- [Training-only preprocessing and leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [Model persistence and unsafe pickle loading](https://scikit-learn.org/stable/model_persistence.html)
- [NumPy random-stream compatibility conditions](https://numpy.org/doc/stable/reference/random/compatibility.html)

**Boundary:** this packaged regression exercise must not be used for patient decisions. We make no medical feature, treatment, clinical efficacy or causal claims. Reproducibility here concerns the frozen computation, not real-world validity.